# AI Avatar Narrator — Reenactment Edition

Upload **four things**:
1. `user_image` — a clear photo of the face you want to animate
2. `voice_sample` — 10–30s clean recording of the voice you want the avatar to speak in
3. `background_image` — the background to composite the avatar onto
4. `driving_video` — the *different* video whose speech content the avatar should repeat

The notebook will:
- Pull the audio out of `driving_video`
- Convert that audio's voice into **your** voice, keeping the exact same words/timing (voice conversion, not text-to-speech)
- Cut your face out of `user_image` and place it on `background_image`
- Lip-sync that composited image to the converted audio with Wav2Lip

**Run cells top to bottom, one at a time. Do not use "Run all"** — there is one required
runtime restart after Cell 1 (numpy version conflict between `rembg`/`TTS` and Colab's
preinstalled packages).

## 1. Install dependencies

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!pip install -q "coqui-tts==0.27.5" rembg "moviepy==1.0.3" pydub

!apt-get -qq install -y ffmpeg > /dev/null

# Clone Wav2Lip for lip-syncing
!git clone -q https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!pip install -q opencv-python numba
!mkdir -p checkpoints
!wget -q "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth" -O checkpoints/wav2lip_gan.pth
%cd ..


Install GFPGAN

In [ ]:
%cd /content
!rm -rf LivePortrait
!git clone -q https://github.com/KwaiVGI/LivePortrait.git
%cd LivePortrait
!pip install -q -r requirements.txt
!pip install -q -U huggingface_hub

from huggingface_hub import snapshot_download
snapshot_download(repo_id="KwaiVGI/LivePortrait", local_dir="./pretrained_weights")
%cd /content


**Now restart the runtime:** `Runtime -> Restart session`.
After it restarts, come back and continue from Cell 2 below — do NOT re-run Cell 1.

## 2. Upload your four files

In [ ]:
from google.colab import files
!pip install -q "transformers==4.57.6"

print("1/5 — Upload your face photo:")
uploaded = files.upload()
USER_IMAGE_PATH = "/content/" + list(uploaded.keys())[0]

print("2/5 — Upload your voice sample (10-30s, clean speech, low background noise):")
uploaded = files.upload()
VOICE_SAMPLE_PATH = "/content/" + list(uploaded.keys())[0]

print("3/5 — Upload your background image:")
uploaded = files.upload()
BACKGROUND_IMAGE_PATH = "/content/" + list(uploaded.keys())[0]

print("4/5 — Upload the driving video (the video whose speech your avatar should repeat):")
uploaded = files.upload()
DRIVING_VIDEO_PATH = "/content/" + list(uploaded.keys())[0]

print("5/5 — Upload the foreground prop image (e.g. mic, transparent PNG):")
uploaded = files.upload()
PROP_IMAGE_PATH = "/content/" + list(uploaded.keys())[0]

print()
print("USER_IMAGE_PATH      =", USER_IMAGE_PATH)
print("VOICE_SAMPLE_PATH    =", VOICE_SAMPLE_PATH)
print("BACKGROUND_IMAGE_PATH=", BACKGROUND_IMAGE_PATH)
print("DRIVING_VIDEO_PATH   =", DRIVING_VIDEO_PATH)
print("PROP_IMAGE_PATH      =", PROP_IMAGE_PATH)

## 3. Pull the audio track out of the driving video

In [ ]:
import subprocess

DRIVING_AUDIO_PATH = "/content/driving_audio.wav"
subprocess.run(
    ["ffmpeg", "-y", "-i", DRIVING_VIDEO_PATH, "-ac", "1", "-ar", "22050", DRIVING_AUDIO_PATH],
    check=True,
)
print("Extracted driving audio ->", DRIVING_AUDIO_PATH)


## 4. Voice conversion — keep the words, change the voice

This does **not** need a transcript or text-to-speech. It takes the driving video's audio
(what's being said, and exactly when) and re-synthesizes it in your voice, using
`voice_sample` as the reference. Timing stays identical to the driving video.

In [ ]:
import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

vc_model = TTS(model_name="voice_conversion_models/multilingual/vctk/freevc24", progress_bar=False).to(device)

CLONED_AUDIO_PATH = "/content/cloned_audio.wav"
vc_model.voice_conversion_to_file(
    source_wav=DRIVING_AUDIO_PATH,
    target_wav=VOICE_SAMPLE_PATH,
    file_path=CLONED_AUDIO_PATH,
)
print("Voice-converted audio saved ->", CLONED_AUDIO_PATH)


## 5. Cut your face out of the photo (background removal)

In [ ]:
!pip uninstall -y cupy-cuda12x

In [ ]:
!pip install -q onnxruntime
from rembg import remove
from PIL import Image

user_img = Image.open(USER_IMAGE_PATH).convert("RGBA")
foreground = remove(user_img)

FOREGROUND_PATH = "/content/user_foreground.png"
foreground.save(FOREGROUND_PATH)
foreground

## 6. Composite your face onto the background image

In [ ]:
bg = Image.open(BACKGROUND_IMAGE_PATH).convert("RGBA").resize(foreground.size)
composited = Image.alpha_composite(bg, foreground).convert("RGB")

COMPOSITED_IMAGE_PATH = "/content/composited_avatar.png"
composited.save(COMPOSITED_IMAGE_PATH)
composited


In [ ]:
ANIMATED_VIDEO_PATH = "/content/animated_avatar.mp4"

!cd LivePortrait && python inference.py \
  -s "{COMPOSITED_IMAGE_PATH}" \
  -d "{DRIVING_VIDEO_PATH}" \
  -o "/content/liveportrait_out" \
  --flag_do_crop \
  --scale 2.3 \
  --vy_ratio -0.05

import glob
result_files = glob.glob("/content/liveportrait_out/**/*.mp4", recursive=True)
result_files = [f for f in result_files if "concat" not in f.lower()]
print("Generated:", result_files)
ANIMATED_VIDEO_PATH = result_files[0]


## 7. Turn the composited still image into a video matching the audio length

In [ ]:
from moviepy.editor import ImageClip, AudioFileClip

audio_clip = AudioFileClip(CLONED_AUDIO_PATH)
duration = audio_clip.duration
print(f"Cloned narration length: {duration:.1f}s")

img_clip = ImageClip(COMPOSITED_IMAGE_PATH).set_duration(duration).set_fps(25)

COMPOSITED_VIDEO_PATH = "/content/composited_avatar.mp4"
img_clip.write_videofile(COMPOSITED_VIDEO_PATH, codec="libx264", audio=False, fps=25)


## 8. Lip-sync with Wav2Lip

In [ ]:
!sed -i 's/librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,/' Wav2Lip/audio.py

In [ ]:
WAV2LIP_OUTPUT_PATH = "/content/lipsynced_output.mp4"

!cd Wav2Lip && python inference.py \
  --checkpoint_path checkpoints/wav2lip_gan.pth \
  --face "{ANIMATED_VIDEO_PATH}" \
  --audio "{CLONED_AUDIO_PATH}" \
  --outfile "{WAV2LIP_OUTPUT_PATH}" \
  --pads 0 15 0 0 \
  --nosmooth

Quality enhancement

---



In [ ]:
%cd /content
!git clone -q https://github.com/TencentARC/GFPGAN.git
%cd GFPGAN
!pip install -q -r requirements.txt
!pip install -q realesrgan
!python setup.py develop -q
!wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -P experiments/pretrained_models
%cd /content

In [ ]:
!find / -name "degradations.py" -path "*basicsr*" 2>/dev/null

In [ ]:
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

In [ ]:
import subprocess, os, json

FRAMES_DIR = "/content/frames_raw"
os.makedirs(FRAMES_DIR, exist_ok=True)

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0",
     "-show_entries", "stream=r_frame_rate", "-of", "json", WAV2LIP_OUTPUT_PATH],
    capture_output=True, text=True
)
fps_str = json.loads(probe.stdout)["streams"][0]["r_frame_rate"]
num, den = map(int, fps_str.split("/"))
ACTUAL_FPS = num / den
print("Detected fps:", ACTUAL_FPS)

subprocess.run(["ffmpeg", "-y", "-i", WAV2LIP_OUTPUT_PATH, f"{FRAMES_DIR}/frame_%05d.png"], check=True)

!cd GFPGAN && python inference_gfpgan.py -i {FRAMES_DIR} -o /content/gfpgan_out -v 1.4 -s 2 --bg_upsampler realesrgan

RESTORED_FRAMES_DIR = "/content/gfpgan_out/restored_imgs"
ENHANCED_NOAUDIO = "/content/enhanced_noaudio.mp4"
subprocess.run(
    ["ffmpeg", "-y", "-framerate", str(ACTUAL_FPS), "-i", f"{RESTORED_FRAMES_DIR}/frame_%05d.png",
     "-c:v", "libx264", "-pix_fmt", "yuv420p", ENHANCED_NOAUDIO],
    check=True,
)

ENHANCED_OUTPUT_PATH = "/content/lipsynced_output_enhanced.mp4"
subprocess.run(
    ["ffmpeg", "-y", "-i", ENHANCED_NOAUDIO, "-i", WAV2LIP_OUTPUT_PATH,
     "-c:v", "copy", "-c:a", "aac", "-map", "0:v:0", "-map", "1:a:0",
     "-shortest", ENHANCED_OUTPUT_PATH],
    check=True,
)
print("Enhanced video saved ->", ENHANCED_OUTPUT_PATH)

In [ ]:
import subprocess

CROPPED_FINAL_PATH = "/content/final_face_neck_only.mp4"
subprocess.run(
    ["ffmpeg", "-y", "-i", ENHANCED_OUTPUT_PATH,
     "-vf", "crop=iw:ih*0.95:0:0",
     "-c:a", "copy", CROPPED_FINAL_PATH],
    check=True
)
print("Saved ->", CROPPED_FINAL_PATH)

In [ ]:
from google.colab import output
from IPython.display import display, HTML
from PIL import Image
import base64, io

import subprocess
subprocess.run(["ffmpeg", "-y", "-i", CROPPED_FINAL_PATH, "-frames:v", "1", "/content/crop_preview_frame.png"], check=True)
base_img = Image.open("/content/crop_preview_frame.png").convert("RGBA")
prop_img = Image.open(PROP_IMAGE_PATH).convert("RGBA")

def _to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

base_b64 = _to_b64(base_img)
prop_b64 = _to_b64(prop_img)

DISPLAY_W = min(base_img.width, 480)
DISPLAY_H = int(base_img.height * DISPLAY_W / base_img.width)
INIT_W = int(DISPLAY_W * 0.4)

def save_mic_position(x_frac, y_frac, width_frac):
    global MIC_X_FRAC, MIC_Y_FRAC, MIC_WIDTH_FRAC
    MIC_X_FRAC, MIC_Y_FRAC, MIC_WIDTH_FRAC = x_frac, y_frac, width_frac
    prop_width = max(1, int(base_img.width * width_frac))
    ratio = prop_width / prop_img.width
    prop_height = int(prop_img.height * ratio)
    prop_resized = prop_img.resize((prop_width, prop_height))
    pos_x = int(base_img.width * x_frac)
    pos_y = int(base_img.height * y_frac)
    preview = base_img.copy()
    preview.paste(prop_resized, (pos_x, pos_y), prop_resized)
    display(preview)
    print(f"Saved -> x_frac={x_frac:.3f}, y_frac={y_frac:.3f}, width_frac={width_frac:.3f}")
    return "ok"

output.register_callback('notebook.save_mic_position', save_mic_position)

html = f"""
<div id="stage" style="position:relative; width:{DISPLAY_W}px; height:{DISPLAY_H}px;
     border:1px solid #999; overflow:visible; background:#eee;">
  <img src="data:image/png;base64,{base_b64}"
       style="width:100%; height:100%; display:block; user-select:none; pointer-events:none;">
  <img id="micImg" src="data:image/png;base64,{prop_b64}"
       style="position:absolute; left:{int(DISPLAY_W*0.3)}px; top:{int(DISPLAY_H*0.6)}px;
              width:{INIT_W}px; cursor:move; touch-action:none;">
</div>
<div style="margin-top:10px; font-family:sans-serif;">
  Mic size: <input type="range" id="sizeSlider" min="20" max="{int(DISPLAY_W*1.5)}" value="{INIT_W}" style="width:250px;">
</div>
<button id="confirmBtn" style="margin-top:10px; padding:8px 16px; font-size:14px; cursor:pointer; position:relative; z-index:10;">  ✅ Confirm mic placement
</button>
<p style="font-family:sans-serif; color:#555; font-size:13px;">
  Drag the mic anywhere. Use the slider to resize. Click Confirm when it looks right.
</p>

<script>
(function() {{
  const mic = document.getElementById('micImg');
  const slider = document.getElementById('sizeSlider');
  let dragging = false, offsetX = 0, offsetY = 0;

  mic.addEventListener('pointerdown', (e) => {{
    dragging = true;
    offsetX = e.clientX - mic.offsetLeft;
    offsetY = e.clientY - mic.offsetTop;
    mic.setPointerCapture(e.pointerId);
  }});
  mic.addEventListener('pointermove', (e) => {{
    if (!dragging) return;
    mic.style.left = (e.clientX - offsetX) + 'px';
    mic.style.top = (e.clientY - offsetY) + 'px';
  }});
  mic.addEventListener('pointerup', () => {{ dragging = false; }});

  slider.addEventListener('input', () => {{
    mic.style.width = slider.value + 'px';
  }});

  document.getElementById('confirmBtn').addEventListener('click', () => {{
    const xFrac = mic.offsetLeft / {DISPLAY_W};
    const yFrac = mic.offsetTop / {DISPLAY_H};
    const widthFrac = mic.offsetWidth / {DISPLAY_W};
    google.colab.kernel.invokeFunction('notebook.save_mic_position', [xFrac, yFrac, widthFrac], {{}});
  }});
}})();
</script>
"""

display(HTML(html))

In [ ]:
from moviepy.editor import VideoFileClip, ImageClip, CompositeVideoClip
from PIL import Image

final_clip = VideoFileClip(CROPPED_FINAL_PATH)
prop_img = Image.open(PROP_IMAGE_PATH).convert("RGBA")

x_frac, y_frac, width_frac = MIC_X_FRAC, MIC_Y_FRAC, MIC_WIDTH_FRAC
prop_width = int(final_clip.w * width_frac)
ratio = prop_width / prop_img.width
prop_height = int(prop_img.height * ratio)
prop_resized = prop_img.resize((prop_width, prop_height))
prop_resized.save("/content/mic_resized.png")

mic_clip = (ImageClip("/content/mic_resized.png")
            .set_duration(final_clip.duration)
            .set_position((int(final_clip.w * x_frac), int(final_clip.h * y_frac))))

FINAL_WITH_MIC_PATH = "/content/final_with_mic.mp4"
CompositeVideoClip([final_clip, mic_clip]).write_videofile(
    FINAL_WITH_MIC_PATH, codec="libx264", audio_codec="aac"
)
print("Saved ->", FINAL_WITH_MIC_PATH)

## 9. Download the result

In [ ]:
from google.colab import files
files.download(FINAL_WITH_MIC_PATH)

---
### Quick reference: what to do if something breaks

- **`ModuleNotFoundError` / numpy `ImportError` right after Cell 1**: the required restart
  after Cell 1 hasn't happened yet. `Runtime -> Restart session`, then continue from Cell 2
  — do not re-run Cell 1.
- **`ImportError: cannot import name 'ImageClip' from 'moviepy.editor'`**: you have moviepy
  2.x installed somewhere, which removed the `editor` submodule. Cell 1 pins
  `moviepy==1.0.3`; if you still see this, restart the runtime and re-run Cell 1 first, before
  anything else installs a newer moviepy over it.
- **Wav2Lip says "Using cpu for inference"**: GPU isn't active. `Runtime -> Change runtime
  type -> GPU -> Save -> Restart` -> redo from Cell 1.
- **Wav2Lip `ValueError: Face not detected`**: the composited image has no clearly visible
  face (bad crop from rembg, face too small/angled, or covered by the background). Check the
  image printed in Cell 6 and reshoot/recrop `user_image` if needed.
- **`voice_conversion_to_file` sounds robotic or barely like your voice**: use a longer, cleaner
  `voice_sample` (20-30s, single speaker, minimal background noise/music).
- **`fatal: destination path 'Wav2Lip' already exists`**: harmless, ignore it — it just means
  you already cloned it in this session.
- **pip dependency conflict warnings** (numba, cuml, jax, etc. mentioning version mismatches):
  harmless noise from unrelated pre-installed Colab packages — ignore unless the cell also
  raises an actual Python error/traceback.